# 3.4 Multi-label Classification

這份 Notebook 使用合成多標籤資料，示範多個 sigmoid 輸出與 binary crossentropy。

## 1. 載入套件

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.datasets import make_multilabel_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, f1_score

tf.keras.utils.set_random_seed(42)

## 2. 建立多標籤資料

每筆資料可能同時擁有多個 label，因此 `y` 是 0/1 矩陣。

In [ ]:
X, y = make_multilabel_classification(
    n_samples=3000,
    n_features=20,
    n_classes=5,
    n_labels=2,
    allow_unlabeled=False,
    random_state=42
)
X = X.astype('float32')
y = y.astype('float32')
label_names = [f'label_{i}' for i in range(y.shape[1])]

print('X shape:', X.shape)
print('y shape:', y.shape)
pd.DataFrame(y[:8], columns=label_names)

## 3. 切分資料與標準化

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train).astype('float32')
x_test = scaler.transform(x_test).astype('float32')

pd.DataFrame(y_train, columns=label_names).mean().to_frame('positive_ratio')

## 4. 建立多標籤 DNN

輸出層有 5 個 sigmoid，每個 label 各自輸出成立機率。

In [ ]:
n_labels = y_train.shape[1]
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(x_train.shape[1],)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(n_labels, activation='sigmoid')
])
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['binary_accuracy']
)
model.summary()

## 5. 訓練模型

In [ ]:
history = model.fit(
    x_train, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=32,
    verbose=0,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)]
)
print('實際訓練 epochs:', len(history.history['loss']))

## 6. 評估模型

多標籤任務會將每個輸出機率用 threshold 轉為 0/1。

In [ ]:
prob = model.predict(x_test, verbose=0)
y_pred = (prob >= 0.5).astype(int)

print(classification_report(y_test, y_pred, target_names=label_names, digits=4, zero_division=0))
print('micro f1:', round(f1_score(y_test, y_pred, average='micro'), 4))
print('macro f1:', round(f1_score(y_test, y_pred, average='macro'), 4))

## 7. 預測新資料

In [ ]:
preview = pd.DataFrame(prob[:5], columns=[f'{name}_prob' for name in label_names])
for i, name in enumerate(label_names):
    preview[f'{name}_pred'] = y_pred[:5, i]
preview

## 8. 小結

多標籤分類的關鍵是：輸出層使用多個 sigmoid，loss 使用 binary crossentropy，評估時觀察每個 label 的 precision、recall 與 F1。